# 02 — GARCH(1,1) (weekly silver volatility)

GARCH(1,1) is the classic parametric benchmark for conditional volatility. Unlike
HAR-RV — which regresses *realised* volatility on its own lags — GARCH models the
volatility of the **return series itself**, so it provides an econometric reference
point of a different kind.

The weekly return series and the RV target both come from `volatility_weekly.csv`,
built in `00_features.ipynb` — run that notebook first.


## Setup


In [1]:
import sys, os
sys.path.append(os.path.abspath('../../src'))

import numpy as np
import pandas as pd
from arch import arch_model
from vol_utils import vol_evaluate, vol_period_metrics, vol_diebold_mariano
from eval_utils import PERIODS
import warnings; warnings.filterwarnings('ignore')

frame = pd.read_csv('../../data/processed/volatility_weekly.csv',
                    parse_dates=['Date']).set_index('Date')
test_df = frame[frame['split'] == 'test']

y_test     = test_df['target'].values
prev_test  = test_df['rv_w_lag1'].values
weekly_ret = frame['silver_ret']               # weekly silver log-returns, full sample
print(f'test weeks: {len(test_df)}')
print(y_test[:5])
print(prev_test[:5])
print(weekly_ret[:5])


test weeks: 175
[0.03579865 0.03010313 0.02336985 0.02631828 0.05362404]
[0.02768859 0.03579865 0.03010313 0.02336985 0.02631828]
Date
2015-04-03   -0.021697
2015-04-10   -0.019120
2015-04-17   -0.009144
2015-04-24   -0.037051
2015-05-01    0.030246
Name: silver_ret, dtype: float64


## 1. Walk-forward GARCH(1,1)

The model assumes

$$r_t = \mu + \varepsilon_t, \qquad \varepsilon_t = \sigma_t z_t, \qquad
\sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2.$$

The one-step-ahead conditional volatility $\hat\sigma_t$ is used as the prediction for
$\text{RV}_t$. To avoid look-ahead the model is **re-fit walking forward week by
week**: at each test week the GARCH is estimated only on weekly returns strictly prior
to that week, then forecast one step ahead. Returns are scaled by 100 during fitting
for numerical stability, and the forecast is scaled back afterwards.


In [2]:
garch_preds = []
for d in test_df.index:
    hist = weekly_ret.loc[:d].iloc[:-1]                            # returns strictly before week d
    m   = arch_model(hist * 100, mean='Constant', vol='GARCH', p=1, q=1, dist='normal')
    res = m.fit(disp='off', show_warning=False)
    fc  = res.forecast(horizon=1, reindex=False)
    garch_preds.append(np.sqrt(fc.variance.values[-1, 0]) / 100)   # back to return scale
garch_preds = np.array(garch_preds)
print(f'fitted {len(garch_preds)} walk-forward GARCH models')


fitted 175 walk-forward GARCH models


## 2. Evaluation


In [3]:
results = [vol_evaluate('GARCH(1,1)', y_test, garch_preds, prev_test)]


GARCH(1,1)                      RMSE=0.03296  MAE=0.01824  R2=+0.249  DCA=0.674


## 3. DM vs the Naïve floor — does GARCH(1,1) genuinely beat $\text{RV}_{t-1}$?

GARCH models the volatility of the *return* series, so unlike HAR-RV its prediction is
a parametric conditional standard deviation rather than a regression on past RV. The
test below asks the same headline question — is this prediction lower-loss than the
Naïve $\text{RV}_{t-1}$ floor under Diebold-Mariano (1995) with Newey-West (1987) lag-1
variance via `vol_diebold_mariano`? Negative DM = GARCH has lower loss.

**QLIKE is the primary loss.** Weekly silver RV is heavy-tailed enough that under
squared-error loss a handful of extreme weeks dominate the differential and inflate the
DM variance, so an RMSE improvement that is real and steady can still fail an MSE-DM
test. The volatility-forecasting literature (Patton 2011) reports forecasts under
**QLIKE**, a proxy-robust ratio loss; squared-error DM is kept underneath only as a
reference. The same comparison is collected across models in `evaluation.ipynb` §4.


In [4]:
# Diebold-Mariano: GARCH(1,1) vs the Naive RV_{t-1} floor.
print('QLIKE loss  --  primary test:')
vol_diebold_mariano(y_test, garch_preds, prev_test, 'GARCH(1,1)', 'Naive', loss='qlike')

print('\nSquared-error loss  --  reference:')
vol_diebold_mariano(y_test, garch_preds, prev_test, 'GARCH(1,1)', 'Naive', loss='mse');


QLIKE loss  --  primary test:
GARCH(1,1)                   vs Naive         [qlike]  DM=-2.594  p=0.009  **    -> winner: GARCH(1,1)

Squared-error loss  --  reference:
GARCH(1,1)                   vs Naive         [mse  ]  DM=-1.161  p=0.246  (ns)  -> winner: tie


## 4. Sub-period breakdown

RMSE and DCA split by calendar year, using the shared `PERIODS` definition.


In [5]:
period_garch = vol_period_metrics(y_test, garch_preds, prev_test, test_df.index, PERIODS)
period_garch.to_csv('../../data/processed/period_garch_volatility.csv')
period_garch.round(4)


,n,RMSE,MAE,DCA
Period,,,,
2023 (choppy),52,0.0162,0.0140,0.6731
2024 (bull start),52,0.0145,0.0122,0.7885
2025 (bull run),52,0.0251,0.0195,0.6346
2026 (YTD),19,0.0836,0.0431,0.4737
── Full test ──,175,0.0330,0.0182,0.6743


## 5. Save outputs

- `metrics_garch_volatility.csv` — GARCH(1,1) headline metrics
- `pred_garch_volatility.csv` — test-set predictions, consumed by `evaluation.ipynb`


In [6]:
pd.DataFrame(results).to_csv('../../data/processed/metrics_garch_volatility.csv', index=False)

pred_garch = pd.DataFrame({'actual': y_test, 'prev': prev_test, 'garch': garch_preds},
                          index=test_df.index)
pred_garch.to_csv('../../data/processed/pred_garch_volatility.csv', index_label='Date')
print('Saved metrics_garch_volatility.csv + pred_garch_volatility.csv')
pd.DataFrame(results).round(5)


Saved metrics_garch_volatility.csv + pred_garch_volatility.csv


,model,rmse,mae,r2,dca
0,"GARCH(1,1)",0.03296,0.01824,0.24899,0.67429
